# 03 — Convert the trained dense model into an MoE, and keep training

**The trainer's ask.** *Train a Linear model and convert that into an MoE! Must show they continue to train and reduce loss.*

## Recipe: sparse upcycling

1. Take the **trained dense** checkpoint from notebook 01.
2. For every `moe_every`-th block, replace the MLP by `E` experts, each initialised as an **exact copy** of that MLP.
   Embeddings, attention, norms and the remaining dense MLPs are copied unchanged.
3. Add a freshly initialised, small-weight **router**.
4. Continue training with a fresh optimiser, a gentle LR re-warm-up and the load-balancing loss.

Because gates are renormalised over the selected experts, and all experts compute the same function, the converted model's output
equals the dense model's for any router. **Loss at the instant of conversion = loss of the dense model**, and
from there the model has `E×` the MLP capacity in the MoE layers to exploit.

To make the claim measurable we run two continuations from the *same* checkpoint on the *same* batches with the *same* LR schedule:

| run | what happens after step `dense_steps` |
|---|---|
| `dense_continue` | **control**: keep training the dense model |
| `moe_upcycled` | convert to MoE, then train |

In [ ]:
import sys, os, math, time, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.environ.get("MOE_SRC", os.path.abspath("../src")))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from moe.config import get_config
from moe.utils import set_seed, get_device, savefig, plot_style, aim_repo, work_root, ckpt_dir, results_dir, is_smoke
from moe.data import load_dataset_tokens
from moe.model import GPT, GPTConfig, MoEMLP, param_report
from moe.runs import dense_config, moe_config, train_config, flops_per_token
from moe.trainer import train, load_history, load_model, lr_at
from moe.upcycle import upcycle, max_logit_diff
from moe.viz import ema, color, heatmap_load
from moe.analysis import make_decoder, make_encoder

plot_style()
X = get_config()
DEVICE = get_device()
torch.set_float32_matmul_precision("high")
set_seed(X.seed)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

print(f"torch {torch.__version__} | device = {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)" if DEVICE == "cuda" else ""))
print(f"preset = {X.name!r}   (override with MOE_PRESET=smoke|small|base)")
print(f"Aim repo : {aim_repo()}   -> browse with:  uv run aim up")
print()
print(pd.Series(X.to_dict(), name="value").to_string())

## 1. Load the trained dense model

In [ ]:
data = load_dataset_tokens(X.dataset)
dense_path = os.path.join(ckpt_dir(), "dense.pt")
assert os.path.exists(dense_path), "run notebook 01 first (it writes checkpoints/dense.pt)"
dense = load_model(dense_path, DEVICE)
H_dense = load_history("dense")
print(f"loaded dense model: {param_report(dense)['total']/1e6:.2f}M params, trained {X.dense_steps} steps")
print(f"its final validation loss: {H_dense['final_val_loss']:.4f}")

## 2. Convert, then *prove* the conversion is lossless

In [ ]:
up = upcycle(dense, n_experts=X.n_experts, top_k=X.top_k, moe_every=X.moe_every, seed=X.seed)
print("upcycle info:", up.upcycle_info)
print("\nblock-by-block:")
for i, b in enumerate(up.transformer.h):
    kind = f"MoE  ({X.n_experts} experts, top-{X.top_k})" if b.is_moe else "dense MLP (copied)"
    print(f"  block {i}: {kind}")

rd, ru = param_report(dense), param_report(up)
print(pd.DataFrame({"dense": {"total (M)": rd["total"]/1e6, "active/token (M)": rd["active"]/1e6, "MFLOPs/token": flops_per_token(dense)/1e6},
                    "upcycled": {"total (M)": ru["total"]/1e6, "active/token (M)": ru["active"]/1e6, "MFLOPs/token": flops_per_token(up)/1e6}}).round(2).to_string())

xb, yb = data.get_batch("val", X.batch_size, X.block_size, DEVICE, torch.Generator().manual_seed(0))
dense.eval(); up.eval()
torch.set_float32_matmul_precision("highest")      # no TF32, so the comparison is true fp32
with torch.no_grad():
    ld, loss_d, _ = dense(xb, yb); lu, loss_u, aux = up(xb, yb)
torch.set_float32_matmul_precision("high")
print(f"\nloss  dense = {loss_d.item():.6f}   upcycled = {loss_u.item():.6f}   |Δ| = {abs(loss_d.item()-loss_u.item()):.2e}")
print(f"max |logit difference| = {(ld-lu).abs().max().item():.2e}   (fp32 round-off -> function preserved)")
print("router at init is ~uniform: mean load per expert", np.round(aux['load'].mean(0).cpu().numpy(), 3).tolist(), "(fair share", 1/X.n_experts, ")")

In [ ]:
# A cheaper variant adds noise to each expert copy to break the symmetry. What does that cost at the instant of conversion?
rows = []
for noise in (0.0, 0.01, 0.05, 0.1):
    u = upcycle(dense, X.n_experts, X.top_k, X.moe_every, expert_noise=noise, seed=X.seed)
    u.eval()
    with torch.no_grad(): _, l, _ = u(xb, yb)
    rows.append({"expert_noise (x weight std)": noise, "loss at conversion": l.item(), "Δ vs dense": l.item() - loss_d.item()})
    del u
print(pd.DataFrame(rows).round(6).to_string(index=False))

## 3. Continue training: `moe_upcycled` vs the `dense_continue` control

Learning rate re-warms to `cont_lr` (lower than the pre-training peak) and decays by cosine. Optimiser state is reset — the MoE has new
parameters, and for the dense control we do the same, to keep the comparison fair. Batches are identical in both runs.

In [ ]:
tc_m = train_config(X, "moe_upcycled", "moe_upcycled", X.cont_steps, X.cont_lr, X.cont_warmup, step_offset=X.dense_steps, device=DEVICE)
up = upcycle(load_model(dense_path, DEVICE), X.n_experts, X.top_k, X.moe_every, seed=X.seed)
up, H_up = train(up, data, tc_m, experiment="session14", aim_repo=aim_repo(),
                 extra_hparams={"arch": "moe", "init": "upcycled", "expert_noise": 0.0, "source": "dense"})

In [ ]:
tc_d = train_config(X, "dense_continue", "dense_continue", X.cont_steps, X.cont_lr, X.cont_warmup, step_offset=X.dense_steps, device=DEVICE)
dc = load_model(dense_path, DEVICE)
dc, H_dc = train(dc, data, tc_d, experiment="session14", aim_repo=aim_repo(),
                 extra_hparams={"arch": "dense", "init": "continued", "source": "dense"})

## 4. Does the converted model keep learning?

In [ ]:
H_up, H_dc, H_dense = load_history("moe_upcycled"), load_history("dense_continue"), load_history("dense")
split = X.dense_steps

fig, axs = plt.subplots(1, 3, figsize=(18, 4.6))
ax = axs[0]
ax.plot(H_dense["step"], ema(H_dense["loss"], 0.1), color=color("dense"), alpha=0.9, label="dense pre-training (train, EMA)")
ax.plot(H_dense["eval_step"], H_dense["val_loss"], "o-", color=color("dense"), ms=3, label="dense pre-training (val)")
for lab, H in (("dense_continue", H_dc), ("moe_upcycled", H_up)):
    ax.plot(H["step"], ema(H["loss"], 0.1), color=color(lab), alpha=0.4)
    ax.plot(H["eval_step"], H["val_loss"], "o-", color=color(lab), ms=3, label=f"{lab} (val)")
ax.axvline(split, color="k", ls="--", lw=1); ax.text(split, ax.get_ylim()[1] * 0.97, " conversion", va="top", fontsize=9)
ax.set_title("full history"); ax.set_xlabel("global step"); ax.set_ylabel("loss"); ax.legend(fontsize=8)
ax.set_ylim(top=min(8, max(H_dense["loss"])))

ax = axs[1]
for lab, H in (("dense_continue", H_dc), ("moe_upcycled", H_up)):
    ax.plot(H["eval_step"], H["val_loss"], "o-", color=color(lab), label=f"{lab} val")
    ax.plot(H["eval_step"], H["train_loss"], "s--", color=color(lab), alpha=0.5, ms=3, label=f"{lab} train")
ax.set_title("after the conversion (zoom)"); ax.set_xlabel("global step"); ax.legend(fontsize=8)

ax = axs[2]
dv = np.array(H_up["val_loss"]) - np.array(H_dc["val_loss"])
ax.bar(H_up["eval_step"], dv, width=max(1, X.eval_interval * 0.7), color=["#2ca02c" if v < 0 else "#d62728" for v in dv])
ax.axhline(0, color="k", lw=0.6); ax.set_title("val-loss gap: MoE − dense_continue (negative = MoE better)"); ax.set_xlabel("global step")
plt.tight_layout(); savefig(fig, "03_upcycle_vs_dense_continue.png"); plt.show()

tab = pd.DataFrame({"global step": H_up["eval_step"], "dense_continue val": H_dc["val_loss"], "moe_upcycled val": H_up["val_loss"]})
tab["MoE − dense"] = tab["moe_upcycled val"] - tab["dense_continue val"]
print(tab.round(4).to_string(index=False))

In [ ]:
v0, v1 = H_up["val_loss"][0], H_up["val_loss"][-1]
print(f"dense model val loss before conversion  : {H_dense['final_val_loss']:.4f}")
print(f"MoE val loss at the moment of conversion: {v0:.4f}   (Δ = {v0 - H_dense['final_val_loss']:+.5f}  -> lossless)")
print(f"MoE val loss after {X.cont_steps} more steps    : {v1:.4f}   (improved by {v0 - v1:.4f} nats, ppl {math.exp(v0):.2f} -> {math.exp(v1):.2f})")
print(f"dense_continue val loss after the same  : {H_dc['final_val_loss']:.4f}   (improved by {H_dc['val_loss'][0] - H_dc['final_val_loss']:.4f})")
print(f"MoE vs dense control                    : {v1 - H_dc['final_val_loss']:+.4f} nats")
monotone = all(b <= a + 0.02 for a, b in zip(H_up["val_loss"], H_up["val_loss"][1:]))
print("\nMoE kept reducing the loss after conversion:", v1 < v0, "| (near-)monotone over evals:", monotone)

## 5. What changed inside the experts?

All experts start as *identical* copies. If they stay identical the MoE is just an expensive dense model.
Two diagnostics: cosine similarity between experts' weights (1 = still clones), and how far each expert has drifted from the original dense MLP.

In [ ]:
from moe.analysis import expert_similarity, expert_drift
sims = expert_similarity(up)
drift = expert_drift(up, load_model(dense_path, DEVICE))
fig, axs = plt.subplots(1, len(sims) + 1, figsize=(4.2 * (len(sims) + 1), 4))
for i, s in enumerate(sims):
    im = axs[i].imshow(s, vmin=min(0.9, s.min()), vmax=1); axs[i].set_title(f"MoE layer {i}: expert cosine sim"); axs[i].grid(False)
plt.colorbar(im, ax=axs[len(sims) - 1])
for li, d in enumerate(drift): axs[-1].bar(np.arange(len(d)) + li * 0.2, d, width=0.2, label=f"MoE layer {li}")
axs[-1].set_title("relative drift from the dense MLP"); axs[-1].set_xlabel("expert"); axs[-1].legend(fontsize=8)
plt.tight_layout(); savefig(fig, "03_expert_divergence.png"); plt.show()
for i, s in enumerate(sims):
    off = s[~np.eye(len(s), dtype=bool)]
    print(f"MoE layer {i}: mean off-diagonal cosine similarity {off.mean():.6f} (min {off.min():.6f})  | mean drift {drift[i].mean():.4f}")

## 6. Routing after conversion

In [ ]:
load = np.array(H_up["load"]); E = load.shape[2]
fig, axs = plt.subplots(1, 4, figsize=(20, 4.2))
im = heatmap_load(axs[0], load[0], "load at conversion"); plt.colorbar(im, ax=axs[0])
im = heatmap_load(axs[1], load[-1], "load at the end"); plt.colorbar(im, ax=axs[1])
ent = np.array(H_up["load_entropy"])
for li in range(ent.shape[1]): axs[2].plot(H_up["eval_step"], ent[:, li], "o-", label=f"MoE layer {li}")
axs[2].set_title("load entropy"); axs[2].set_xlabel("global step"); axs[2].legend(fontsize=8); axs[2].set_ylim(0, 1.02)
axs[3].plot(H_up["step"], H_up["lb"], color="#2ca02c"); axs[3].axhline(1, color="k", ls="--", lw=0.6)
axs[3].set_title("load-balancing loss (1 = uniform)"); axs[3].set_xlabel("global step")
plt.tight_layout(); savefig(fig, "03_upcycled_routing.png"); plt.show()
print("dead experts over time:", H_up["dead_experts"])

## 7. (Optional) Ablations

Off by default (each is a full continuation run). Enable with `MOE_ABLATIONS=1` (always on in the smoke test).
All of them start from the same dense checkpoint and see the same batches.

In [ ]:
RUN_ABLATIONS = is_smoke() or os.environ.get("MOE_ABLATIONS") == "1"
print("running ablations:", RUN_ABLATIONS)
if RUN_ABLATIONS:
    variants = {
        "noise0.02":  dict(up=dict(expert_noise=0.02)),
        "no_lb_loss": dict(tc=dict(lb_coef=0.0)),
        "top1":       dict(up=dict(top_k=1)),
        "experts16":  dict(up=dict(n_experts=2 * X.n_experts)),
    }
    abl = {"moe_upcycled (reference)": H_up, "dense_continue (control)": H_dc}
    for name, v in variants.items():
        kw = dict(n_experts=X.n_experts, top_k=X.top_k, moe_every=X.moe_every, seed=X.seed); kw.update(v.get("up", {}))
        m = upcycle(load_model(dense_path, DEVICE), **kw)
        t = train_config(X, f"abl_{name}", f"abl_{name}", X.cont_steps, X.cont_lr, X.cont_warmup, step_offset=X.dense_steps, device=DEVICE, **v.get("tc", {}))
        _, h = train(m, data, t, experiment="session14_ablations", aim_repo=aim_repo(), extra_hparams={"ablation": name, **v.get("up", {})})
        abl[name] = h
    rows = {k: {"val @ conversion": h["val_loss"][0], "final val": h["final_val_loss"], "Δ vs dense_continue": h["final_val_loss"] - H_dc["final_val_loss"]} for k, h in abl.items()}
    print(pd.DataFrame(rows).T.round(4).to_string())
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for k, h in abl.items(): ax.plot(h["eval_step"], h["val_loss"], "o-", ms=3, label=k)
    ax.set_xlabel("global step"); ax.set_ylabel("val loss"); ax.legend(fontsize=8); ax.set_title("ablations")
    plt.tight_layout(); savefig(fig, "03_ablations.png"); plt.show()